In [ ]:
############## Code to get balance sheet table  and PnL data #####################
import pdfplumber
from PyPDF2 import PdfWriter, PdfReader
import json
import os

# Helper: check if a string is a number
# Handles negative numbers, decimals, and commas
# -----------------------------
def is_number(s):
    try:
        float(s.replace(',', ''))
        return True
    except ValueError:
        return False
# -----------------------------
# :one: Extract company info
# -----------------------------


input_dir = "pdfs\\CFS"

for filename in os.listdir(input_dir):
    if filename.endswith(".pdf"):
        print("filename :", filename)
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_16_08_2025.pdf"
        # input_pdf_path =r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_05_09_2025 - Copy.pdf"
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_16_08_2025.pdf"
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_30_08_2025.pdf"
        input_pdf_path = r"D:\\Nexensus_Projects\\pdfFoms\\pdfs\\CFS\\" + filename
        # reader = PdfReader(input_pdf_path)
        # writer = PdfWriter()

        # -----------------------------
        # Extract all lines from PDF
        # -----------------------------
        all_lines = []
        with pdfplumber.open(input_pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                # print("page_num", page_num)
                for line in page.extract_text_lines():
                    page_line_list = [line['text'] for line in page.extract_text_lines()]
                    # print(any("Figures as at the".lower() in item.lower() for item in page_line_list))

                    ######################### For getting Balance sheet page numbers ###########################################
                    if "Part B - Balance Sheet" in line['text'] or "Consolidated Balance Sheet" in line['text']:
                        if any("Figures as at the".lower() in item.lower() for item in page_line_list):
                            print("starting page of balance sheet", page_num + 1)
                            # page_line_list = [line['text'] for line in page.extract_text_lines()]
                            # print(any("Figures as at the".lower() in item.lower() for item in page_line_list))
                            BS_starting_page = page_num + 1
                        else:
                            BS_starting_page = page_num + 2
                            print("starting page of balance sheet", page_num + 2)
                    if "TOTAL LIABILITIES" in line['text']:
                        print("ending page of balance sheet", page_num+1)
                        BS_ending_page = page_num +1 


                    ##################### For getting profit and loss page numbers ##########################################
                    if "Statement of Profit and Loss".lower() in line['text'].lower() or "Statement of Consolidated Profit and Loss".lower() in line['text'].lower():
                        if any("Figures for the period".lower() in item.lower() for item in page_line_list):
                            print("starting page of profit and loss", page_num + 1)
                            # page_line_list = [line['text'] for line in page.extract_text_lines()]
                            # print(any("Figures as at the".lower() in item.lower() for item in page_line_list))
                            PnL_starting_page = page_num + 1
                        else:
                            PnL_starting_page = page_num + 2
                            print("starting page of profit and loss", page_num + 2)
                    # if "Financial parameters - Profit and loss".lower() in line['text'].lower():
                    if "Earnings per equity share (for continuing and discontinuing operations)".lower() in line['text'].lower():  
                        print("ending page of profit and loss", page_num+1)
                        PnL_ending_page = page_num +1    

        # all_rows = []
        with pdfplumber.open(input_pdf_path) as pdf:
            all_rows_BS = []
            all_rows_PnL = []
            for page_num, page in enumerate(pdf.pages):
                if page_num + 1 >= BS_starting_page and page_num + 1 <=BS_ending_page: 
                    tables = page.extract_tables()
                    print("page_num :", page_num + 1, len(tables))
                    for table in tables:
                        # print(len(table[0]), table[0])
                        if page_num +1 == BS_starting_page: 
                            BS_header = table[0]
                            for t in table[1:]:
                                if len(t) == 7:
                                    print(len(t), t)
                                    # all_rows.append({
                                    #     "Particulars" : t[1],
                                    #     "Figures as at the\nend of (Current\nreporting period)\n(in Rs.)\n31/03/2025\n(DD/MM/YYYY)" : t[2],
                                    #     "Figures as at the\nend of (Previous\nreporting period)\n(in Rs.)\n31/03/2024\n(DD/MM/YYYY)" : t[3],
                                    #     "Reason for\nchange in\npre-filled\nfigures of\nprevious\nreporting\nperiod" : t[4],
                                    #     "Figures as at the\nbeginning of\n(Previous reporting\nperiod) (in Rs.)\n(DD/MM/YYYY)" : t[5],
                                    #     "Reason for\nchange in\npre-filled\nfigures of\nprevious\nreporting\nperiod"  : t[6],
                                    #     })
                                    all_rows_BS.append({
                                        BS_header[1] : t[1],
                                        BS_header[2] : t[2],
                                        BS_header[3] : t[3],
                                        # BS_header[4] : t[4],
                                        # BS_header[5] : t[5],
                                        # BS_header[6]  : t[6],
                                        })
                        else:
                            for t in table:
                                if len(t) == 7:
                                    print(len(t), t)
                                    # all_rows.append({
                                    #     "Particulars" : t[1],
                                    #     "Figures as at the\nend of (Current\nreporting period)\n(in Rs.)\n31/03/2025\n(DD/MM/YYYY)" : t[2],
                                    #     "Figures as at the\nend of (Previous\nreporting period)\n(in Rs.)\n31/03/2024\n(DD/MM/YYYY)" : t[3],
                                    #     "Reason for\nchange in\npre-filled\nfigures of\nprevious\nreporting\nperiod" : t[4],
                                    #     "Figures as at the\nbeginning of\n(Previous reporting\nperiod) (in Rs.)\n(DD/MM/YYYY)" : t[5],
                                    #     "Reason for\nchange in\npre-filled\nfigures of\nprevious\nreporting\nperiod"  : t[6],
                                    #     })
                                    all_rows_BS.append({
                                        BS_header[1] : t[1],
                                        BS_header[2] : t[2],
                                        BS_header[3] : t[3],
                                        # BS_header[4] : t[4],
                                        # BS_header[5] : t[5],
                                        # BS_header[6]  : t[6],
                                        })
                

                if page_num + 1 >= PnL_starting_page and page_num + 1 <=PnL_ending_page: 
                    tables = page.extract_tables()
                    print("page_num :", page_num + 1, len(tables))
                    if page_num + 1 == PnL_starting_page and len(tables) >=2:
                        tables = tables[len(tables)-1:]
                    if page_num + 1 == PnL_ending_page and len(tables) >=2:
                        tables = tables[:1]
                    for table in tables:
                        # print(len(table[0]), table[0])
                        if page_num +1 == PnL_starting_page: 
                            PnL_header = table[1]
                            for t in table[2:]:
                                # print("coming here", len(t), t[0])
                                if len(t) == 7:
                                    print(len(t), t)
                                    all_rows_PnL.append({
                                        PnL_header[2] : t[2],
                                        PnL_header[3] : t[3],
                                        PnL_header[4] : t[4],
                                        # PnL_header[4] : t[4],
                                        # PnL_header[5] : t[5],
                                        # PnL_header[6]  : t[6],
                                        })
                        else:
                            # print("tables in page :", len(table), table)
                            for t in table:
                                # print("coming here", len(t), t[0])
                                if len(t) == 7:
                                    print(len(t), t)
                                    all_rows_PnL.append({
                                        PnL_header[2] : t[2],
                                        PnL_header[3] : t[3],
                                        PnL_header[4] : t[4],
                                        # PnL_header[4] : t[4],
                                        # PnL_header[5] : t[5],
                                        # PnL_header[6]  : t[6],
                                        })

                                    # if t[0] is None or t[0] == '':
                                    #     print(len(t), t)
                                    #     all_rows_PnL.append({
                                    #         PnL_header[2] : t[2],
                                    #         PnL_header[3] : t[3],
                                    #         PnL_header[4] : t[4],
                                    #         # PnL_header[4] : t[4],
                                    #         # PnL_header[5] : t[5],
                                    #         # PnL_header[6]  : t[6],
                                    #         })
                                    # else:
                                    #     print(len(t), t)
                                    #     all_rows_PnL.append({
                                    #         PnL_header[2] : t[2],
                                    #         PnL_header[3] : t[3],
                                    #         PnL_header[4] : t[4],
                                    #         # PnL_header[4] : t[4],
                                    #         # PnL_header[5] : t[5],
                                    #         # PnL_header[6]  : t[6],
                                    #         })

filename : AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_16_08_2025.pdf
starting page of balance sheet 4
ending page of balance sheet 6
starting page of profit and loss 16
ending page of profit and loss 18
page_num : 4 1
7 ['I', 'ASSETS', None, None, None, None, None]
7 ['(1)', 'Financial Assets', None, None, None, None, None]
7 ['', '(a) Cash and cash\nequivalents', '20666306143', '24698665291', '', '', '']
7 ['', '(b) Bank Balance\nother than (a)\nabove', '21251466904', '17758507272', '', '', '']
7 ['', '(c) Derivative\nfinancial\ninstruments', '502497204', '1576863135', '', '', '']
7 ['', '(d) Receivables', None, None, None, None, None]
7 ['', '(I) Trade\nReceivables', '1075259665', '1024228512', '', '', '']
7 ['', '(II) Other\nReceivables', '0', '296523275', '', '', '']
7 ['', 'e) Loans', '553642574647', '509523153683', '', '', '']
7 ['', '(f) Investments', '44379871563', '40589828580', '', '', '']
7 ['', '(g) Other Financial\nAssets', '11429746262', '14207706911', 'Regrouping for\nbett

In [12]:
# Step 3: Convert all rows to DataFrame (reset index safely)
import pandas as pd
if all_rows_BS:
    BS_df = pd.DataFrame(all_rows_BS).reset_index(drop=True)
    PnL_df = pd.DataFrame(all_rows_PnL).reset_index(drop=True)
    print("✅ Extracted rows BS:", len(BS_df))
    print("✅ Extracted rows PnL:", len(PnL_df))

    # Optional: Save output for each file
    output_path = os.path.join(input_dir, f"BalanceSheet.xlsx")
    BS_df.to_excel(output_path, index=False)
    print("💾 Saved:", output_path)

    # Optional: Save output for each file
    output_path = os.path.join(input_dir, f"PnL.xlsx")
    PnL_df.to_excel(output_path, index=False)
    print("💾 Saved:", output_path)
else:
    print("⚠️ No valid table rows found in", filename)

✅ Extracted rows BS: 52
✅ Extracted rows PnL: 66
💾 Saved: pdfs\CFS\BalanceSheet.xlsx
💾 Saved: pdfs\CFS\PnL.xlsx


In [2]:
import pdfplumber
import os
from xml.etree import ElementTree as ET
from html import escape
def extract_text_and_tables(file_path):
    """
    Extracts text and tables from a PDF file.
    Returns a dictionary with text lines and tables.
    """
    data = {"text": [], "tables": []}
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                # Extract text
                text = page.extract_text()
                if text:
                    data["text"].extend([line.strip() for line in text.split('\n') if line.strip()])
                # Extract tables
                tables = page.extract_tables()
                for table in tables:
                    if table:
                        data["tables"].append([[escape(str(cell)) if cell else "" for cell in row] for row in table])
        return data
    except Exception as e:
        raise ValueError(f"Error processing PDF: {str(e)}")
def pdf_to_html(file_path, output_path):
    """
    Converts PDF content to HTML format and saves it.
    """
    data = extract_text_and_tables(file_path)
    html_content = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>PDF to HTML Conversion</title>
    <style>
        table { border-collapse: collapse; width: 100%; }
        th, td { border: 1px solid black; padding: 8px; text-align: left; }
        th { background-color: #F2F2F2; }
    </style>
</head>
<body>
    <h1>PDF Content</h1>
    <div class="text-content">"""
    # Add text content
    for line in data["text"]:
        html_content += f"<p>{escape(line)}</p>\n"
    html_content += "</div>\n"
    # Add tables
    for i, table in enumerate(data["tables"], 1):
        html_content += f"<h2>Table {i}</h2>\n<table>\n"
        for row in table:
            html_content += "<tr>\n"
            for cell in row:
                html_content += f"<td>{cell}</td>\n"
            html_content += "</tr>\n"
        html_content += "</table>\n"
    html_content += "</body>\n</html>"
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html_content)
def pdf_to_xml(file_path, output_path):
    """
    Converts PDF content to XML format and saves it.
    """
    data = extract_text_and_tables(file_path)
    root = ET.Element("document")
    root.set("filename", os.path.basename(file_path))
    # Add text content
    text_node = ET.SubElement(root, "text_content")
    for line in data["text"]:
        line_node = ET.SubElement(text_node, "line")
        line_node.text = escape(line)
    # Add tables
    for i, table in enumerate(data["tables"], 1):
        table_node = ET.SubElement(root, "table", id=f"table_{i}")
        for j, row in enumerate(table, 1):
            row_node = ET.SubElement(table_node, "row", id=f"row_{j}")
            for k, cell in enumerate(row, 1):
                cell_node = ET.SubElement(row_node, "cell", id=f"cell_{k}")
                cell_node.text = cell
    tree = ET.ElementTree(root)
    tree.write(output_path, encoding="utf-8", xml_declaration=True)
# Example usage
folder_path = r"D:\Nexensus_Projects\pdfFoms\pdfs\CFS"
for filename in os.listdir(folder_path):
    if filename.lower().endswith('.pdf'):
        input_pdf_path = os.path.join(folder_path, filename)
        # Convert to HTML
        html_output_path = os.path.join(folder_path, f"{os.path.splitext(filename)[0]}.html")
        pdf_to_html(input_pdf_path, html_output_path)
        print(f"Converted {filename} to {os.path.basename(html_output_path)}")
        # Convert to XML
        xml_output_path = os.path.join(folder_path, f"{os.path.splitext(filename)[0]}.xml")
        pdf_to_xml(input_pdf_path, xml_output_path)
        print(f"Converted {filename} to {os.path.basename(xml_output_path)}")








Converted AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_16_08_2025.pdf to AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_16_08_2025.html
Converted AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_16_08_2025.pdf to AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_16_08_2025.xml


In [16]:
############## Code to get balance sheet table  and PnL data #####################
import pdfplumber
from PyPDF2 import PdfWriter, PdfReader
import json
import os

# Helper: check if a string is a number
# Handles negative numbers, decimals, and commas
# -----------------------------
def is_number(s):
    try:
        float(s.replace(',', ''))
        return True
    except ValueError:
        return False
# -----------------------------
# :one: Extract company info
# -----------------------------


input_dir = "pdfs\\CFS"

for filename in os.listdir(input_dir):
    if filename.endswith(".pdf"):
        print("filename :", filename)
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_16_08_2025.pdf"
        # input_pdf_path =r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_05_09_2025 - Copy.pdf"
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_16_08_2025.pdf"
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_30_08_2025.pdf"
        input_pdf_path = r"D:\\Nexensus_Projects\\pdfFoms\\pdfs\\CFS\\" + filename
        # reader = PdfReader(input_pdf_path)
        # writer = PdfWriter()

        # -----------------------------
        # Extract all lines from PDF
        # -----------------------------
        all_lines = []
        with pdfplumber.open(input_pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                # print("page_num", page_num)
                # for line in page.extract_text_lines():
                #     print(line['text'])
                for x in page.extract_words():
                    print(x['text'], "::::::::::::::::::::::", x['x1'])

filename : AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_12_09_2025.pdf
Form :::::::::::::::::::::: 58.730000000000004
No. :::::::::::::::::::::: 82.39
AOC-4 :::::::::::::::::::::: 122.29
CFS :::::::::::::::::::::: 145.90800000000002
NBFC :::::::::::::::::::::: 179.98400000000004
(Ind :::::::::::::::::::::: 206.29
AS) :::::::::::::::::::::: 228.928
1-20968482848_SRN_FORM_1757673452947 :::::::::::::::::::::: 527.91
Form :::::::::::::::::::::: 495.028
language :::::::::::::::::::::: 541.864
Form :::::::::::::::::::::: 50.07
for :::::::::::::::::::::: 64.42
filing :::::::::::::::::::::: 87.33
consolidated :::::::::::::::::::::: 142.98999999999998
financial :::::::::::::::::::::: 180.59
statements :::::::::::::::::::::: 229.74
and :::::::::::::::::::::: 247.68
other :::::::::::::::::::::: 51.22
documents :::::::::::::::::::::: 99.77000000000001
with :::::::::::::::::::::: 120.78000000000002
the :::::::::::::::::::::: 136.91000000000003
Registrar :::::::::::::::::::::: 176.53000000000003
English

In [4]:
x.keys()

dict_keys(['text', 'x0', 'x1', 'top', 'doctop', 'bottom', 'upright', 'height', 'width', 'direction'])

In [13]:
x["direction"]

'ltr'